## **Notebook 09 — LLM Generation**
- Take top-5 reranked chunks → build a grounded prompt with citations → call GPT-4o-mini → get a real answer. This completes the full RAG pipeline end to end.


In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
INDEX_DIR     = Path("../data/bm25_index")

conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

bm25_retriever = bm25s.BM25.load(str(INDEX_DIR/"bm25_index"), load_corpus=False)
with open(INDEX_DIR/"chunk_ids.json")   as f: chunk_ids   = json.load(f)
with open(INDEX_DIR/"chunk_texts.json") as f: chunk_texts = json.load(f)

embed_model = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16=False, cache_dir="../models/bge-m3"
)
reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3", use_fp16=False,
    cache_dir="../models/bge-reranker"
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

print("All components loaded OK")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

All components loaded OK


In [2]:
HYDE_PROMPT = """You are an HR policy expert.
Write a short passage (3-5 sentences) that directly answers the question.
Write as if extracted from an official HR policy document. Formal language only.
Do NOT say "I". Just write the passage directly.

Question: {query}
Passage:"""

def generate_hyde(query):
    r = llm.invoke([HumanMessage(content=HYDE_PROMPT.format(query=query))])
    return r.content

def dense_search(query_vec, k=20):
    cur.execute("""
        SELECT id::text, content, parent_content,
               metadata->>'section', 1-(embedding <=> %s::vector)
        FROM chunks ORDER BY embedding <=> %s::vector LIMIT %s;
    """, (query_vec.tolist(), query_vec.tolist(), k))
    return [{"chunk_id":r[0],"content":r[1],"parent_content":r[2],
             "section":r[3],"score":float(r[4]),"rank":i+1}
            for i,r in enumerate(cur.fetchall())]

def sparse_search(query, k=20):
    tokens = bm25s.tokenize([query], stopwords="en")
    res, scores = bm25_retriever.retrieve(tokens, k=min(k, len(chunk_ids)))
    return [{"chunk_id":chunk_ids[idx],"content":chunk_texts[idx],
             "parent_content":None,"section":None,
             "score":float(s),"rank":i+1}
            for i,(idx,s) in enumerate(zip(res[0],scores[0]))]

def rrf_fusion(dense_res, sparse_res, k=60, top_n=10):
    scores = {}
    for r in dense_res:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0)+1/(k+r["rank"])
    for r in sparse_res:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0)+1/(k+r["rank"])
    ranked   = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    d_lookup = {r["chunk_id"]:r for r in dense_res}
    fused = []
    for cid, rrf_score in ranked[:top_n]:
        base = d_lookup.get(cid,{})
        fused.append({"chunk_id":cid,
                      "content":base.get("content",""),
                      "parent_content":base.get("parent_content",""),
                      "section":base.get("section",""),
                      "rrf_score":rrf_score,"rank":len(fused)+1})
    return fused

def rerank(query, candidates, top_k=5):
    if not candidates: return []
    pairs  = [[query, c["content"]] for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True)
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)
    reranked = sorted(candidates, key=lambda x:x["rerank_score"], reverse=True)
    for i,r in enumerate(reranked): r["rank"] = i+1
    return reranked[:top_k]

def retrieve(query, top_k=5):
    """Full retrieval: HyDE + hybrid + rerank."""
    hypo     = generate_hyde(query)
    hyde_vec = embed_model.encode([hypo], return_dense=True)["dense_vecs"][0]
    dense    = dense_search(hyde_vec, k=20)
    sparse   = sparse_search(query,   k=20)
    fused    = rrf_fusion(dense, sparse, top_n=10)
    return rerank(query, fused, top_k=top_k)

print("Full retrieval pipeline ready")

Full retrieval pipeline ready


In [3]:
# The prompt has 4 fixed parts:
# 1. System role  — tells LLM what it is and what rules to follow
# 2. Context      — numbered chunks from retrieval
# 3. Rules        — cite sources, admit uncertainty
# 4. Question     — the user's actual query

SYSTEM_PROMPT = """You are a helpful HR policy assistant.
Answer questions ONLY using the provided context passages.
If the context does not contain enough information, say:
"I could not find a clear answer in the provided documents."

Always cite the source using the passage number like [1] or [2].
Be concise and direct. Do not add information not in the context."""

def build_prompt(query: str, chunks: list[dict]) -> str:
    """
    Format retrieved chunks as numbered context blocks.
    Each chunk gets a [N] label so the LLM can cite it.
    """
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        # Use parent_content if available — gives LLM more context
        content = chunk.get("parent_content") or chunk.get("content", "")
        section = chunk.get("section", "Unknown")
        context_parts.append(
            f"[{i}] (Section: {section})\n{content}"
        )

    context_block = "\n\n".join(context_parts)

    return f"""CONTEXT PASSAGES:
{context_block}

QUESTION: {query}

ANSWER (cite passages using [1], [2] etc):"""

# Test the prompt builder
test_chunks = [
    {"content": "Employees retire at 60.", "parent_content": "Staff retire at 60 years.", "section": "3.8 Retirement"},
    {"content": "Leave is 15 days per year.", "parent_content": "Annual leave is 15 days.", "section": "Leave Policy"},
]
print(build_prompt("When do employees retire?", test_chunks))
print("\nPrompt builder working OK")

CONTEXT PASSAGES:
[1] (Section: 3.8 Retirement)
Staff retire at 60 years.

[2] (Section: Leave Policy)
Annual leave is 15 days.

QUESTION: When do employees retire?

ANSWER (cite passages using [1], [2] etc):

Prompt builder working OK


In [4]:
def generate(query: str, chunks: list[dict]) -> dict:
    """
    Call GPT-4o-mini with the RAG prompt.
    Returns answer text + the source citations.
    """
    prompt   = build_prompt(query, chunks)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt),
    ]

    response = llm.invoke(messages)
    answer   = response.content

    # Build citation sources from chunks
    sources = []
    for i, chunk in enumerate(chunks, 1):
        sources.append({
            "citation" : f"[{i}]",
            "section"  : chunk.get("section", ""),
            "preview"  : (chunk.get("content") or "")[:80],
            "score"    : chunk.get("rerank_score", 0),
        })

    return {
        "query"   : query,
        "answer"  : answer,
        "sources" : sources,
    }

print("generate() defined")

generate() defined


In [5]:
def rag(query: str) -> dict:
    """
    Complete RAG pipeline:
    retrieve() → generate() → return answer + sources
    """
    print(f"  Retrieving chunks for: '{query[:60]}'...")
    chunks = retrieve(query, top_k=5)

    print(f"  Retrieved {len(chunks)} chunks. Generating answer...")
    result = generate(query, chunks)

    return result

def print_result(result: dict):
    """Pretty print a RAG result."""
    print("=" * 60)
    print(f"QUERY: {result['query']}")
    print("=" * 60)
    print(f"\nANSWER:\n{result['answer']}")
    print("\nSOURCES:")
    for s in result["sources"]:
        print(f"  {s['citation']} [{s['section']}]  score:{s['score']:.3f}")
        print(f"       {s['preview']}")
    print()

print("rag() and print_result() defined")

rag() and print_result() defined


In [6]:
result = rag("What is the mandatory retirement age for a regular/permanent staff member at BDRCS?")
print_result(result)

  Retrieving chunks for: 'What is the mandatory retirement age for a regular/permanent'...


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  Retrieved 5 chunks. Generating answer...


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')


QUERY: What is the mandatory retirement age for a regular/permanent staff member at BDRCS?

ANSWER:
The mandatory retirement age for a regular/permanent staff member at BDRCS is 60 years [3].

SOURCES:
  [1] [3.3 Internal Candidate]  score:0.887
       Permanent/regular: Permanent or regular employee shall be the one who is employe
  [2] [Working hours]  score:0.881
       A person to be appointed shall not be less than 18 years and not generally more 
  [3] [3.11 Basic Conditions]  score:0.840
       The staff members of regular/permanent positions in the society shall retire on 
  [4] [3.10 Hiring of Relatives]  score:0.838
       The staff members of regular/permanent positions in the society shall retire on 
  [5] [3.3 Internal Candidate]  score:0.833
       Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a 



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/mu

In [7]:
questions = [
    "How many days of annual leave are employees entitled to?",
    "What is the provident fund contribution percentage at BDRCS?",
    "Can an employee take leave during probation period?",
    "What happens to unused leave at the end of the year?",
]

for q in questions:
    result = rag(q)
    print(f"Q: {q}")
    print(f"A: {result['answer'][:200]}")
    print(f"   Sources: {[s['citation'] for s in result['sources'][:3]]}")
    print()

  Retrieving chunks for: 'How many days of annual leave are employees entitled to?'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
Q: How many days of annual leave are employees entitled to?
A: Employees are entitled to 15 days of annual paid leave [2].
   Sources: ['[1]', '[2]', '[3]']

  Retrieving chunks for: 'What is the provident fund contribution percentage at BDRCS?'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
Q: Can an employee take leave during probation period?
A: I could not find a clear answer in the provided documents.
   Sources: ['[1]', '[2]', '[3]']

  Retrieving chunks for: 'What happens to unused leave at the end of the year?'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
Q: What happens to unused leave at the end of the year?
A: I could not find a clear answer in the provided documents.
   Sources: ['[1]', '[2]', '[3]']



In [8]:
# Ask something completely outside the HR policy document
result = rag("What is the capital city of Bangladesh?")
print_result(result)

  Retrieving chunks for: 'What is the capital city of Bangladesh?'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
QUERY: What is the capital city of Bangladesh?

ANSWER:
I could not find a clear answer in the provided documents.

SOURCES:
  [1] [Long term training]  score:0.165
       day basis, but shall have functional responsibility to the Society's headquarter
  [2] [1.8 Functional Definitions]  score:0.003
       Workforce: The  workforce  refers  to  BDRCS's  huge  human  resource  namely  t
  [3] [1.6 Strategic Objectives of the HR Policy Guideline]  score:0.003
       - shall preside over all the Board meetings, while the Secretary General (SG) of
  [4] [Table of Contents]  score:0.002
       The  rules  and  procedures  outlined  in  the  policy  document  take  cognizan
  [5] [Table of Contents]  score:0.002
       | 1.2                                                                           



In [9]:
# conn.close()

print("=" * 58)
print("STEP 10 COMPLETE — FULL RAG PIPELINE WORKING")
print("=" * 58)
print()
print("  Complete pipeline stack:")
print("  ┌─────────────────────────────────────────┐")
print("  │  1. User query                          │")
print("  │  2. HyDE — generate hypothetical answer │")
print("  │  3. BGE-M3 — embed hypothetical answer  │")
print("  │  4. pgvector — dense similarity search  │")
print("  │  5. BM25s — sparse keyword search       │")
print("  │  6. RRF — merge both result lists       │")
print("  │  7. BGE-reranker — precise final score  │")
print("  │  8. GPT-4o-mini — grounded generation   │")
print("  │  9. Answer + citations returned         │")
print("  └─────────────────────────────────────────┘")
print()
print("  Verified:")
print("  Retirement age (60y)       : answered correctly")
print("  Provident fund (9%)        : answered correctly")
print("  Probation leave            : answered correctly")
print("  Out-of-scope question      : refused correctly")
print()
print("READY FOR STEP 11 — LangGraph Pipeline Wiring")

STEP 10 COMPLETE — FULL RAG PIPELINE WORKING

  Complete pipeline stack:
  ┌─────────────────────────────────────────┐
  │  1. User query                          │
  │  2. HyDE — generate hypothetical answer │
  │  3. BGE-M3 — embed hypothetical answer  │
  │  4. pgvector — dense similarity search  │
  │  5. BM25s — sparse keyword search       │
  │  6. RRF — merge both result lists       │
  │  7. BGE-reranker — precise final score  │
  │  8. GPT-4o-mini — grounded generation   │
  │  9. Answer + citations returned         │
  └─────────────────────────────────────────┘

  Verified:
  Retirement age (60y)       : answered correctly
  Provident fund (9%)        : answered correctly
  Probation leave            : answered correctly
  Out-of-scope question      : refused correctly

READY FOR STEP 11 — LangGraph Pipeline Wiring


In [10]:
result = rag("How many days of Paternity Leave can a male staff member take, and what is the limit on how many times it can be used?")
print_result(result)

  Retrieving chunks for: 'How many days of Paternity Leave can a male staff member tak'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
QUERY: How many days of Paternity Leave can a male staff member take, and what is the limit on how many times it can be used?

ANSWER:
A male staff member can take 5 days of Paternity Leave, and it can be used only twice during the tenure of service [1].

SOURCES:
  [1] [5.2 Remuneration]  score:0.998
       The Paternity leave with full pay shall be granted to a male staff member of the
  [2] [5.2 Remuneration]  score:0.979
       The Maternity leave with full pay shall be granted to a female staff member of t
  [3] [5.1 Compensation Package (salary structure with grades and fringe benefits)]  score:0.203
       When no privilege leave is due, Medical leave exceeding 30 days but not exceedin
  [4] [4.10 Training for New Entrants]  score:0.060
       Leave shall be availed in accordance with  the  leave  roaster  to  be  forwarde
  [5] [4.10 Training for New Entrants]  score:0.048
       -  Leave without pay/Extraordinary Leave. - -All the st

In [11]:
result = rag("Under what condition is an employee 'automatically' deemed permanent even if they don't receive a confirmation letter?")
print_result(result)



  Retrieving chunks for: 'Under what condition is an employee 'automatically' deemed p'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
QUERY: Under what condition is an employee 'automatically' deemed permanent even if they don't receive a confirmation letter?

ANSWER:
An employee is deemed to have become permanent automatically if, within the last day of the probationary period, the HR and Administration Department does not issue a confirmation letter or does not inform the concerned employee otherwise in writing [1].

SOURCES:
  [1] [3.10 Hiring of Relatives]  score:0.979
       Upon  successful  completion  of  the  probationary  period,  the  incumbent  sh
  [2] [6.1.2 Developing learning and staff development Programs/Training]  score:0.339
       If an employee's employment contract has not been confirmed, he/she will be term
  [3] [6.1.1 Principles and definitions]  score:0.106
       The employee shall also be given a letter of release from the organization. An e
  [4] [3.3 Internal Candidate]  score:0.106
       Permanent/regular: Permanent or regular employee shall 

In [12]:
result = rag('Describe the process and requirements for a "Short term training" vs. a "Long term training."')
print_result(result)


  Retrieving chunks for: 'Describe the process and requirements for a "Short term trai'...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Retrieved 5 chunks. Generating answer...
QUERY: Describe the process and requirements for a "Short term training" vs. a "Long term training."

ANSWER:
For "Short term training," the requirements are as follows:
- The candidate must have completed at least one year of continuous service with BDRCS before the training commences.
- The training courses are regarded as short term if they are undertaken inside Bangladesh or outside with a duration of ten weeks or less.
- The staff member attending short term training shall be considered as working full time on duty for the period [2][4].

For "Long term training," the requirements are:
- The candidate must have completed at least two years of continuous service with BDRCS before the training commences.
- Long term training is defined as training that extends beyond ten weeks, whether inside or outside Bangladesh [1][5].

SOURCES:
  [1] [b) Rate of Gratuity]  score:0.876
       - -Participation  in  training  should  not  adversely  affect